# 02 — Phase-1 Pilot Verification & Automated GO / NO-GO Scorecard

**Scope**: `When2Speak-ValCalib` (200 turns, Invariant #8 calibration) + `When2Speak-Pilot` (400 turns) + `pm_official_8` + 8 mini PageMantle-Ext tasks (4 action + 4 traps) + 1 mini online DB-CSE wave (8 stream tasks).  
**Estimated Budget**: ~\$1–\$2 incremental API spend (Drive SQLite cache reuses Jev, Direct Chat, and Judge calls at \$0).

In [10]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent DRIVE_ROOT: /content/drive/MyDrive/when2speak_research


In [11]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

# Ensure `src/` is immediately importable in a fresh Colab kernel without restarting Python
_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


Already up to date.
/content/when2speak
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


Secret lookup diagnostics: {
  "jev": {
    "status": "found",
    "source": "drive:JEV_API_KEY.txt",
    "masked": "apik...c3aa"
  },
  "openai": {
    "status": "found",
    "source": "drive:OPENAI_KEY.txt",
    "masked": "sk-p...yN8A"
  }
}
Resolved models: {'timestamp_utc': '2026-11-20T13:25:11.705978+00:00', 'luna': 'gpt-4.1-mini', 'terra': 'gpt-4.1', 'sol': 'gpt-6.1-sol', 'jev_model': 'jev-latest', 'jev_healthy': True, 'triage_backend': 'jev', 'available_openai_models_count': 99, 'notes': ["Probe failed for luna candidate 'gpt-6-luna': BadRequestError", "Probe failed for luna candidate 'gpt-5.6-luna': BadRequestError", "Probe failed for terra candidate 'gpt-6.1-terra': BadRequestError", "Probe failed for terra candidate 'gpt-5.6-terra': BadRequestError"]}


## Step 1: Cost Estimate & Run Configuration (`MODE = 'pilot'`)

In [13]:
from datetime import datetime, timezone
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"
PILOT_VERSION = "v3"  # Bump tag when changing Stage-2 prompts so stale predictions aren't replayed
est = estimate_run_cost_usd(mode=MODE, include_pagemantle=True, include_ablations=False)
print("Estimated Pilot API Calls & Cost:", json.dumps(est, indent=2))

RUN_DIR = DRIVE_ROOT / "runs" / f"pilot_{PILOT_VERSION}_{datetime.now(timezone.utc).strftime('%Y%m%d')}"
save_run_metadata(RUN_DIR, {"mode": MODE, "pilot_version": PILOT_VERSION, "estimate": est, "dial_cfg": DIAL_CFG.model_dump()})
print("Run outputs will persist to:", RUN_DIR)


Estimated Pilot API Calls & Cost: {
  "mode": "pilot",
  "estimated_api_calls": 1072,
  "estimated_when2speak_usd": 0.88,
  "estimated_pagemantle_usd": 2.3,
  "estimated_ablations_usd": 0.0,
  "estimated_total_usd": 3.18
}
Run outputs will persist to: /content/drive/MyDrive/when2speak_research/runs/pilot_v3_20261121


## Step 2: Validation Calibration (<=200 turns, Invariant #8) + When2Speak-Pilot (400 turns)

In [14]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test
from dial_skill.eval.runners import calibrate_when2speak_thresholds, run_when2speak_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("gate", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("executor", MODELS_CFG, RESOLVED_MODELS)
triage_backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(
    models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=triage_backend
)

w2s_val = load_when2speak_split("val_calib", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
w2s_pilot = load_when2speak_split("pilot", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])

# 2A. Strictly leakage-free validation-split threshold calibration (<= 200 turns, Invariant #8)
val_agent = DialSkillAgent(
    triage_client=triage_client,
    openai_client=oai,
    gate_model=luna_model,
    executor_model=terra_model,
    skill_store=SkillStore.from_json_file(),
    cfg=DIAL_CFG,
    method_name="dial_skill_luna",
)
w2s_val_eps = await run_when2speak_eval(
    agent=val_agent, turns=w2s_val, output_jsonl=RUN_DIR / "predictions_when2speak_val_calib.jsonl", resume=True
)
cal = calibrate_when2speak_thresholds(
    val_turns=w2s_val,
    val_episodes=w2s_val_eps,
    eval_turns=w2s_pilot,
    lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
    tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
    theta_floor=DIAL_CFG.gate.theta_floor,
    max_fir=0.11,
    max_mir=0.45,
)
print("Validation-split calibration (zero test leakage):", json.dumps(cal, indent=2))
(RUN_DIR / "when2speak_calibration.json").write_text(json.dumps(cal, indent=2), encoding="utf-8")

# 2B. Evaluate When2Speak-Pilot (400 turns) using validation-calibrated (lambda, tau_dm)
w2s_cfg = DIAL_CFG.model_copy(deep=True)
w2s_cfg.gate.lambda_disruption = cal["best_lambda"]
w2s_cfg.gate.tau_dm = cal["best_tau_dm"]
w2s_dial_agent = DialSkillAgent(
    triage_client=triage_client,
    openai_client=oai,
    gate_model=luna_model,
    executor_model=terra_model,
    skill_store=SkillStore.from_json_file(),
    cfg=w2s_cfg,
    method_name="dial_skill_luna",
)
direct_agent = DirectChatBaseline(openai_client=oai, model=luna_model)

w2s_dial_eps = await run_when2speak_eval(
    agent=w2s_dial_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_dial.jsonl", resume=True
)
w2s_direct_eps = await run_when2speak_eval(
    agent=direct_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_direct.jsonl", resume=True
)

y_true = [t.label_speak for t in w2s_pilot]
w2s_dial_m = compute_when2speak_metrics(y_true, w2s_dial_eps)
w2s_direct_m = compute_when2speak_metrics(y_true, w2s_direct_eps)
mc = mcnemar_test(y_true, [e.gate.binary_speak for e in w2s_dial_eps], [e.gate.binary_speak for e in w2s_direct_eps])
print("When2Speak-Pilot DIAL-Skill:", json.dumps(w2s_dial_m, indent=2))
print("When2Speak-Pilot Direct Chat:", json.dumps(w2s_direct_m, indent=2))
print("McNemar Test:", mc)


W2S[predictions_when2speak_val_calib]:   0%|          | 0/200 [00:00<?, ?it/s]

Validation-split calibration (zero test leakage): {
  "best_lambda": 0.5,
  "best_tau_dm": 0.7,
  "val_macro_f1": 0.605951,
  "val_fir": 0.083333,
  "val_mir": 0.7,
  "n_val_turns": 200
}


W2S[predictions_when2speak_dial]:   0%|          | 0/400 [00:00<?, ?it/s]

W2S[predictions_when2speak_direct]:   0%|          | 0/400 [00:00<?, ?it/s]

When2Speak-Pilot DIAL-Skill: {
  "n_samples": 400,
  "tp": 14,
  "tn": 322,
  "fp": 27,
  "fn": 37,
  "macro_f1": 0.606976,
  "macro_f1_ci95": [
    0.540347,
    0.671135
  ],
  "speak_f1": 0.304348,
  "speak_precision": 0.341463,
  "speak_recall": 0.27451,
  "silent_f1": 0.909605,
  "fir": 0.077364,
  "mir": 0.72549,
  "auroc": 0.673352,
  "auprc": 0.275588,
  "ece": 0.47985,
  "margin_gap": 0.203348,
  "token_reduction_pct": 40.51,
  "short_circuit_rate": 0.005,
  "total_cost_usd": 0.423989,
  "cost_per_1k_turns_usd": 1.06
}
When2Speak-Pilot Direct Chat: {
  "n_samples": 400,
  "tp": 36,
  "tn": 141,
  "fp": 208,
  "fn": 15,
  "macro_f1": 0.401242,
  "macro_f1_ci95": [
    0.355797,
    0.449898
  ],
  "speak_f1": 0.244068,
  "speak_precision": 0.147541,
  "speak_recall": 0.705882,
  "silent_f1": 0.558416,
  "fir": 0.595989,
  "mir": 0.294118,
  "auroc": 0.554947,
  "auprc": 0.141647,
  "ece": 0.510074,
  "margin_gap": 0.219788,
  "token_reduction_pct": 0.0,
  "short_circuit_rate": 

## Step 3: PageMantle Pilot (`pm_official_8` + 8 mini traps/actions) + 1 Mini DB-CSE Wave

In [15]:
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_online_evolution_stream, run_pagemantle_eval
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.schemas import PageMantleTask

def _load_pm(name: str):
    p = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_stream_w1 = [t for t in _load_pm("pm_online_stream_24") if t.wave == 1]
pm_eval_all = _load_pm("pm_eval_32")
pm_mini_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [
    t for t in pm_eval_all if t.expected_action == "SILENT"
][:4]
pm_pilot_tasks = pm_official + pm_mini_eval

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
env.ensure_extracted()
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

# Use clean default DIAL_CFG (lambda=1.0, tau_dm=0.05) for PageMantle enterprise environment
pm_dial_agent = DialSkillAgent(
    triage_client=triage_client,
    openai_client=oai,
    gate_model=luna_model,
    executor_model=terra_model,
    skill_store=SkillStore.from_json_file(),
    cfg=DIAL_CFG,
    method_name="dial_skill_luna",
)

pm_dial_eps = await run_pagemantle_eval(
    agent=pm_dial_agent, tasks=pm_pilot_tasks, env=env, output_jsonl=RUN_DIR / "predictions_pm_dial.jsonl", resume=True
)
pm_direct_eps = await run_pagemantle_eval(
    agent=DirectChatBaseline(openai_client=oai, model=terra_model),
    tasks=pm_pilot_tasks,
    env=env,
    output_jsonl=RUN_DIR / "predictions_pm_direct.jsonl",
    resume=True,
)
pm_dial_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_dial_eps)]
pm_direct_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_direct_eps)]

pm_dial_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_dial_eps, pm_dial_scores)
pm_direct_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_direct_eps, pm_direct_scores)
print("PageMantle DIAL-Skill:", json.dumps(pm_dial_m, indent=2))
print("PageMantle Direct Chat:", json.dumps(pm_direct_m, indent=2))

# Mini DB-CSE Wave 1 sanity check (strictly on pm_online_stream_24 Wave 1)
updater = DBCSEUpdater(
    skill_store=pm_dial_agent.skill_store,
    openai_client=oai,
    critic_model=terra_model,
    cfg=DIAL_CFG,
    log_path=RUN_DIR / "skill_evolution_log.jsonl",
)
wave1_summary = await run_online_evolution_stream(
    agent=pm_dial_agent, updater=updater, stream_tasks=pm_stream_w1, env=env, run_dir=RUN_DIR / "mini_wave1", judge=judge
)
pm_post_eps = await run_pagemantle_eval(
    agent=pm_dial_agent, tasks=pm_mini_eval, env=env, output_jsonl=RUN_DIR / "predictions_pm_post_w1.jsonl", resume=False
)
pm_post_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_mini_eval, pm_post_eps)]
pm_pre_mini = compute_pagemantle_metrics(pm_mini_eval, pm_dial_eps[len(pm_official):], pm_dial_scores[len(pm_official):])
pm_post_mini = compute_pagemantle_metrics(pm_mini_eval, pm_post_eps, pm_post_scores)
dbcse_delta = pm_post_mini["overall_score"] - pm_pre_mini["overall_score"]
print("DB-CSE Wave-1 stats:", json.dumps(wave1_summary["dbcse_stats"], indent=2))
print(f"Mini Wave-1 DB-CSE held-out: pre={pm_pre_mini['overall_score']:.4f} -> post={pm_post_mini['overall_score']:.4f} (delta={dbcse_delta:+.4f})")


PM[predictions_pm_dial]:   0%|          | 0/16 [00:00<?, ?it/s]

PM[predictions_pm_direct]:   0%|          | 0/16 [00:00<?, ?it/s]

PageMantle DIAL-Skill: {
  "n_tasks": 16,
  "overall_score": 8.5719,
  "overall_score_ci95": [
    7.3167,
    9.3954
  ],
  "intervention_or_silence": 9.0625,
  "content_and_evidence": 8.2812,
  "clarity_and_tone": 8.4062,
  "audience_and_location": 8.4688,
  "silence_accuracy": 1.0,
  "action_recall": 0.9167,
  "channel_match_rate": 0.6875,
  "deescalation_utility": 0.8562,
  "timeout_rate": 0.0,
  "avg_tool_calls": 2.12,
  "token_reduction_pct": -77.07,
  "total_cost_usd": 0.152934,
  "cost_per_task_usd": 0.009558,
  "per_category_overall": {
    "analyze": 9.375,
    "check": 3.925,
    "coordinate": 8.8875,
    "decide": 8.475,
    "diagnose": 8.075,
    "explain": 9.6083,
    "find": 9.5833,
    "guard": 9.325
  }
}
PageMantle Direct Chat: {
  "n_tasks": 16,
  "overall_score": 5.8781,
  "overall_score_ci95": [
    3.9,
    7.5925
  ],
  "intervention_or_silence": 6.5,
  "content_and_evidence": 5.5625,
  "clarity_and_tone": 6.25,
  "audience_and_location": 5.2188,
  "silence_accur

PM[predictions_pm_post_w1]:   0%|          | 0/8 [00:00<?, ?it/s]

DB-CSE Wave-1 stats: {
  "episodes_processed": 8,
  "numeric_updates": 6,
  "patches_proposed": 1,
  "patches_rejected_guardrail": 0,
  "patches_blocked_escrow": 0,
  "patches_promoted": 1,
  "pruning_events": 0
}
Mini Wave-1 DB-CSE held-out: pre=8.4969 -> post=8.4531 (delta=-0.0438)


## Step 4: Automated Pilot GO / NO-GO Scorecard

In [16]:
from dial_skill.eval.metrics import evaluate_pilot_go_nogo
from dial_skill.utils.checkpoint import load_jsonl

# Measure Stage-1 Jev Triage compression on multi-turn chats (>= 6 turns; When2Speak dialogue windows cap at 8 turns)
long_eps = [e for t, e in zip(w2s_pilot, w2s_dial_eps) if len(t.messages) >= 6]
long_raw = sum(e.prompt_tokens_raw for e in long_eps) or 1
long_slate = sum(e.prompt_tokens_slate for e in long_eps)
long_red = (1.0 - long_slate / long_raw) if long_eps else (w2s_dial_m["token_reduction_pct"] / 100.0)

# True Stage-1 decisive-message recall: fraction of true SPEAK turns NOT dropped/short-circuited by Stage 1
pos_pairs = [(t, e) for t, e in zip(w2s_pilot, w2s_dial_eps) if t.label_speak]
triage_decisive_recall = (
    sum(1 for _, e in pos_pairs if not e.gate.short_circuited) / max(1, len(pos_pairs))
)

# Verify promoted DB-CSE anchor regressions directly from the saved evolution audit log
evo_log_rows = load_jsonl(RUN_DIR / "skill_evolution_log.jsonl") if (RUN_DIR / "skill_evolution_log.jsonl").is_file() else []
promoted_anchor_regs = sum(
    int((row.get("escrow_details") or {}).get("anchor_regressions", 0))
    for row in evo_log_rows
    if row.get("promoted")
)

scorecard = evaluate_pilot_go_nogo(
    w2s_dial=w2s_dial_m,
    w2s_direct=w2s_direct_m,
    pm_dial=pm_dial_m,
    pm_direct=pm_direct_m,
    triage_long_chat_token_reduction=long_red,
    triage_decisive_recall=triage_decisive_recall,
    dbcse_anchor_regressions=promoted_anchor_regs,
    dbcse_heldout_delta=dbcse_delta,
    thresholds=EXP_CFG.get("pilot_go_nogo"),
)
scorecard["triage_compression_details"] = {
    "multi_turn_count": len(long_eps),
    "token_reduction": round(long_red, 4),
    "stage1_decisive_recall": round(triage_decisive_recall, 4),
}
scorecard["triage_meta"] = triage_client.get_run_metadata()
scorecard["total_pilot_usd"] = round(oai.usage.total_cost_usd + triage_client.total_cost_usd, 4)

(RUN_DIR / "pilot_scorecard.json").write_text(json.dumps(scorecard, indent=2), encoding="utf-8")
print("=== PILOT VERIFICATION SCORECARD ===")
print(json.dumps(scorecard, indent=2))


=== PILOT VERIFICATION SCORECARD ===
{
  "verdict": "NO_GO",
  "passed_checks": 3,
  "total_checks": 5,
  "checks": {
    "1_when2speak_pilot_timing": false,
    "2_pagemantle_end_to_end": true,
    "3_jev_triage_compression": true,
    "4_margin_discriminability": false,
    "5_dbcse_sanity": true
  },
  "diagnostics": [
    "MIR too high: lower tau_dm (0.05 -> -0.05), theta_skip (0.15 -> 0.08), or theta_floor."
  ],
  "triage_compression_details": {
    "multi_turn_count": 294,
    "token_reduction": 0.4653,
    "stage1_decisive_recall": 0.9804
  },
  "triage_meta": {
    "configured_triage_backend": "jev",
    "jev_calls": 0,
    "jev_cache_hits": 4139,
    "fallback_count": 0,
    "jev_circuit_open_reason": null,
    "fallback_reasons_sample": [],
    "triage_cost_usd": 0.0
  },
  "total_pilot_usd": 0.8512
}


## Step 5: Zero-Cost Pilot Deep Diagnostics (Reads Saved Drive Artifacts, $0.00)

Inspects When2Speak score distributions, component AUROCs, top FPs/FNs, PageMantle per-task metrics & routing, and DB-CSE evolution logs.

In [17]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.schemas import EpisodeResult, PageMantleTask
from dial_skill.utils.cache import SQLiteAPICache
from dial_skill.utils.checkpoint import load_jsonl

pilot_dirs = sorted((DRIVE_ROOT / "runs").glob("pilot_*"), key=lambda p: p.stat().st_mtime)
DIAG_RUN_DIR = RUN_DIR if "RUN_DIR" in globals() and RUN_DIR.exists() else pilot_dirs[-1]
print("Inspecting saved Pilot Run:", DIAG_RUN_DIR)

# 1. When2Speak Deep Diagnostics
w2s_turns = load_when2speak_split("pilot", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
w2s_dial_rows = [EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_when2speak_dial.jsonl")]
by_id = {e.sample_id: e for e in w2s_dial_rows}
paired = [(t, by_id[t.sample_id]) for t in w2s_turns if t.sample_id in by_id]

pos_adv = [e.gate.advocate_score for t, e in paired if t.label_speak]
neg_adv = [e.gate.advocate_score for t, e in paired if not t.label_speak]
pos_skep = [e.gate.skeptic_score for t, e in paired if t.label_speak]
neg_skep = [e.gate.skeptic_score for t, e in paired if not t.label_speak]
pos_del = [e.gate.margin_delta for t, e in paired if t.label_speak]
neg_del = [e.gate.margin_delta for t, e in paired if not t.label_speak]

print("\n--- When2Speak-Pilot Stage-2 Score Distributions ---")
print(f"SPEAK  (n={len(pos_adv)}): adv={np.mean(pos_adv):.3f}±{np.std(pos_adv):.3f}, skep={np.mean(pos_skep):.3f}±{np.std(pos_skep):.3f}, delta={np.mean(pos_del):.3f}")
print(f"SILENT (n={len(neg_adv)}): adv={np.mean(neg_adv):.3f}±{np.std(neg_adv):.3f}, skep={np.mean(neg_skep):.3f}±{np.std(neg_skep):.3f}, delta={np.mean(neg_del):.3f}")
yt = [int(t.label_speak) for t, _ in paired]
print(f"Component AUROC: adv={roc_auc_score(yt, [e.gate.advocate_score for _, e in paired]):.4f} | -skep={roc_auc_score(yt, [-e.gate.skeptic_score for _, e in paired]):.4f} | margin={roc_auc_score(yt, [e.gate.margin_delta for _, e in paired]):.4f}")
print(f"Avg raw tokens={np.mean([e.prompt_tokens_raw for _,e in paired]):.1f}, avg slate tokens={np.mean([e.prompt_tokens_slate for _,e in paired]):.1f}")

print("\nTop 5 False Positives (label=SILENT, highest margin):")
for t, e in sorted([(t, e) for t, e in paired if not t.label_speak], key=lambda x: x[1].gate.margin_delta, reverse=True)[:5]:
    last = t.messages[-1]
    print(f"  len={len(t.messages)} adv={e.gate.advocate_score:.2f} skep={e.gate.skeptic_score:.2f} | {last.speaker}: {last.text[:130]!r}")

print("\nTop 5 False Negatives (label=SPEAK, lowest margin):")
for t, e in sorted([(t, e) for t, e in paired if t.label_speak], key=lambda x: x[1].gate.margin_delta)[:5]:
    last = t.messages[-1]
    print(f"  len={len(t.messages)} adv={e.gate.advocate_score:.2f} skep={e.gate.skeptic_score:.2f} | {last.speaker}: {last.text[:110]!r} | REF={(t.reference_response or '')[:90]!r}")

# 2. PageMantle Pilot Breakdown + Re-hydrate Cached Dual-Judge Scores from SQLite ($0.00!)
def _load_pm(name: str):
    fp = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in fp.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_eval_all = _load_pm("pm_eval_32")
pm_mini_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [t for t in pm_eval_all if t.expected_action == "SILENT"][:4]
pm_tasks = pm_official + pm_mini_eval
pm_dial_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_dial.jsonl")}
pm_dir_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_direct.jsonl")}
pm_post_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_post_w1.jsonl")}

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
terra_model = resolve_model_for_role("judge", MODELS_CFG, RESOLVED_MODELS)
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

pm_dial_eps = [pm_dial_map[t.task_id] for t in pm_tasks if t.task_id in pm_dial_map]
pm_dir_eps = [pm_dir_map[t.task_id] for t in pm_tasks if t.task_id in pm_dir_map]
pm_post_eps = [pm_post_map[t.task_id] for t in pm_mini_eval if t.task_id in pm_post_map]

pm_dial_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_tasks, pm_dial_eps)]
pm_dir_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_tasks, pm_dir_eps)]
pm_post_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_mini_eval, pm_post_eps)]
print(f"\nRe-hydrated judge scores (new API cost = ${oai.usage.total_cost_usd:.4f}, cache hits = {oai.usage.cache_hits})")

dial_score_map = {s.sample_id: s.overall_score for s in pm_dial_scores}
dir_score_map = {s.sample_id: s.overall_score for s in pm_dir_scores}
post_score_map = {s.sample_id: s.overall_score for s in pm_post_scores}

print("\n--- PageMantle Pilot Per-Task Comparison (16 tasks) ---")
for t in pm_tasks:
    ed = pm_dial_map.get(t.task_id)
    eb = pm_dir_map.get(t.task_id)
    ep_post = pm_post_map.get(t.task_id)
    print(
        f"  {t.task_id:15s} | cat={t.category:10s} | exp={t.expected_action:9s} ({t.ideal_tier.value:17s}) "
        f"| DIAL={ed.gate.final_tier.value if ed else 'NA':17s} ({dial_score_map.get(t.task_id, 0):.1f}, adv={ed.gate.advocate_score:.2f},skep={ed.gate.skeptic_score:.2f},tools={ed.tool_calls_used}) "
        f"| DIRECT={eb.gate.final_tier.value if eb else 'NA':17s} ({dir_score_map.get(t.task_id, 0):.1f}, tools={eb.tool_calls_used})"
        + (f" | POST_W1={ep_post.gate.final_tier.value} ({post_score_map.get(t.task_id, 0):.1f})" if ep_post else "")
    )

# 3. DB-CSE Wave-1 Evolution Log
evo_log_path = DIAG_RUN_DIR / "skill_evolution_log.jsonl"
if evo_log_path.is_file():
    evo_rows = load_jsonl(evo_log_path)
    print(f"\n--- DB-CSE Wave-1 Evolution Log ({len(evo_rows)} entries) ---")
    for row in evo_rows:
        print(f"  {row['sample_id']:18s} | skill={row['skill_name']:10s} | fb={row['feedback_type']:20s} | rc={str(row.get('root_cause')):18s} | promoted={row['promoted']} | reject={row.get('rejection_reason')} | num={row.get('numeric_update')}")


Inspecting saved Pilot Run: /content/drive/MyDrive/when2speak_research/runs/pilot_v3_20261121

--- When2Speak-Pilot Stage-2 Score Distributions ---
SPEAK  (n=51): adv=0.594±0.253, skep=0.452±0.364, delta=0.358
SILENT (n=349): adv=0.491±0.265, skep=0.670±0.315, delta=0.155
Component AUROC: adv=0.6621 | -skep=0.6786 | margin=0.6734
Avg raw tokens=280.7, avg slate tokens=167.0

Top 5 False Positives (label=SILENT, highest margin):
  len=8 adv=0.85 skep=0.10 | Speaker_1: 'Oh wow thats more chemistry than I expected but it makes perfect sense Thanks [AGENT]'
  len=8 adv=0.85 skep=0.10 | Speaker_4: 'That makes a lot of sense [AGENT] It really does put things into perspective Maybe focusing on what I enjoy and finding someone wh'
  len=8 adv=0.85 skep=0.10 | Speaker_3: "It's all about adding value, isn't it Instead of just echoing what's already out there"
  len=8 adv=0.85 skep=0.10 | Speaker_1: 'Yeah Ill make a note to visit or maybe even call them tomorrow to see if they can forward it to u